# Identifying Fast X-ray Transient (FXT) Candidates

### STATiX Variable Source Catalogue — Pouliasis et al. (A&A)

This notebook reproduces the extragalactic Fast X-ray Transient (FXT) candidate search described in the paper, using **only the published catalogue** (`STATIX_VC_published.fits`).



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
from astropy.table import Table
from astropy.time import Time
from astropy.coordinates import SkyCoord
import astropy.units as u
from scipy import signal
from astropy.stats import mad_std as mad

## 1. Load the published catalogue

In [ ]:
out = Table.read("STATIX_VC_published.fits")
print(f"Loaded {len(out)} detections with {len(out.colnames)} columns.")

## 2. Compute Galactic latitude and unified classification

Same reconstructions used in the QPE notebook: `b_deg` from equatorial coordinates, and `final_class` unifying `CLASS` with a `CLASS_RF` fallback.

In [ ]:
coords = SkyCoord(ra=np.asarray(out["RA_X"]) * u.deg,
                   dec=np.asarray(out["DEC_X"]) * u.deg,
                   frame="icrs")
out["b_deg"] = coords.galactic.b.deg

out["final_class"] = np.where(out["CLASS"] == -1, out["CLASS_RF"], out["CLASS"])

print("Galactic latitude and unified classification computed.")

## 3. Reconstruct the pn Bayesian-block count

The published table does not carry a precomputed `PN_n_blocks` column; it is recovered from the number of valid (non-NaN-padded) rows in `pn_LC_BB`.

In [ ]:
def n_blocks(bb_row):
    arr = np.ma.filled(np.atleast_2d(bb_row), np.nan)
    valid = ~np.isnan(arr).all(axis=1)
    return int(valid.sum())


out["pn_n_blocks"] = np.array([n_blocks(row) for row in out["pn_LC_BB"]])
print("pn_n_blocks distribution (first few counts):")
for n in range(0, 6):
    print(f"  {n} blocks: {np.sum(out['pn_n_blocks'] == n)} sources")

## 4. Step 1 — Select single-peak, exactly-3-block sources

Following the paper, we require:
1. `FLAG_SINGLEPEAK` (a single up-transition with no matching down-transition), and
2. exactly 3 pn Bayesian blocks.

In [ ]:
bb3_pn = out["FLAG_SINGLEPEAK_pn"] & (out["pn_n_blocks"] == 3)
print(f"Single-peak, exactly-3-block pn sources: {np.sum(bb3_pn)}")


## 5. Step 2 — Require the isolated-flare block pattern

For the 3-block sources, we require the significance pattern **not-significant / significant / not-significant** across the three blocks -- i.e. an isolated flare with no detection immediately before or after it.

In [ ]:
candidate_rows = out[bb3_pn]
pn_lc_bb = np.array([np.ma.filled(row, np.nan) for row in candidate_rows["pn_LC_BB"]])

mask_bb = (
    (pn_lc_bb[:, 0, 5] == 0) &
    (pn_lc_bb[:, 2, 5] == 0) &
    (pn_lc_bb[:, 1, 5] == 1)
)

flashes = candidate_rows[mask_bb]
print(f"Isolated single-flare candidates (flashes): {len(flashes)}")

## 6. Step 3 — Restrict to extragalactic, previously-unseen sources

We keep sources classified as likely or secure extragalactic (`final_class` in {2, 3}), and require `group_number == 1`, i.e. no prior detections of the same source in other observations.

In [ ]:
flashes_extrag = flashes[(flashes["final_class"] == 2) | (flashes["final_class"] == 3)]
print(f"Extragalactic flashes: {len(flashes_extrag)}")

flashes_extrag_UNI = flashes_extrag[flashes_extrag["group_number"] == 1]
print(f"...with no prior detections elsewhere (group_number == 1): {len(flashes_extrag_UNI)}")

## 7. Compute the central-block (flare) duration

In [ ]:
arr = np.array([np.ma.filled(row, np.nan) for row in flashes_extrag_UNI["pn_LC_BB"]])
duration = arr[:, 1, 1] - arr[:, 1, 0]
flashes_extrag_UNI["pn_duration_sxrt"] = duration

print(f"Median flare duration: {np.median(duration):.1f} s")

## 8. Step 4 — High Galactic latitude cut

In [ ]:
flashes_extrag_UNI_B10 = flashes_extrag_UNI[np.abs(flashes_extrag_UNI["b_deg"]) >= 10]
n_secure = np.sum(flashes_extrag_UNI_B10["final_class"] == 3)

print(f"Final FXT candidate sample: {len(flashes_extrag_UNI_B10)}")
print(f"  of which secure extragalactic (final_class == 3): {n_secure}")
print()


## 9. Optical counterparts



In [ ]:
has_gaia = ~flashes_extrag_UNI_B10["GDR3_ID"].mask if hasattr(flashes_extrag_UNI_B10["GDR3_ID"], "mask") \
    else (flashes_extrag_UNI_B10["GDR3_ID"] > 0)
has_ls10 = flashes_extrag_UNI_B10["OPT_SURVEY"] == "LS10"

has_optical = has_gaia | has_ls10
print(f"FXT candidates with an LS10 or Gaia DR3 counterpart: {np.sum(has_optical)} / {len(flashes_extrag_UNI_B10)}")
print()
print("Redshift information (NED) is not stored as a column in the published "
      "table. The paper notes DETID=108419004010045 has a known redshift "
      "(Seyfert 2, z=0.066) from NED -- this would need a live NED lookup "
      "by position/NED_ID to reproduce from this table alone.")

## 10. Flux, duration and significance diagnostics

The statistical significance of each flare is quantified via the maximum Z-score at the rise/fall transitions of the central block (`pn_Z_SCORE` indices 1 and 2).

In [ ]:
z_scores = np.array([np.ma.filled(np.atleast_1d(row), np.nan) for row in flashes_extrag_UNI_B10["pn_Z_SCORE"]])

plt.figure(figsize=(6, 4.5))
plt.scatter(flashes_extrag_UNI_B10["pn_duration_sxrt"], z_scores[:, 2], label="fall transition (block 2)", alpha=0.7)
plt.scatter(flashes_extrag_UNI_B10["pn_duration_sxrt"], z_scores[:, 1], label="rise transition (block 1)", alpha=0.7)
plt.yscale("log")
plt.xscale("log")
plt.xlabel("Duration [s]")
plt.ylabel("Z-score")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4.5))
plt.scatter(flashes_extrag_UNI_B10["pn_duration_sxrt"], flashes_extrag_UNI_B10["pn_FLUX"], alpha=0.7)
plt.yscale("log")
plt.xscale("log")
plt.xlabel("Duration [s]")
plt.ylabel(r"Flux [erg cm$^{-2}$ s$^{-1}$]")
plt.tight_layout()
plt.show()

In [ ]:
z_max = np.maximum(z_scores[:, 1], z_scores[:, 2])

fig, ax = plt.subplots(figsize=(7, 4.5))
sc = ax.scatter(
    flashes_extrag_UNI_B10["pn_duration_sxrt"],
    flashes_extrag_UNI_B10["pn_FLUX"],
    c=z_max,
    cmap="turbo",
    norm=mcolors.Normalize(vmin=np.nanmin(z_max), vmax=np.nanmax(z_max)),
    edgecolors="black",
    linewidths=0.5,
    alpha=0.8,
    s=50,
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("FXT Candidates", fontsize=14)
ax.set_xlabel("Duration [s]", fontsize=14)
ax.set_ylabel(r"Flux [erg cm$^{-2}$ s$^{-1}$]", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=12)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r"Maximum $Z$-score (Flare Significance)", fontsize=12)
cbar.ax.tick_params(labelsize=11)

plt.tight_layout()
plt.savefig("fxt_flux_duration_zscore.pdf", bbox_inches="tight")
plt.show()

## 12. Light-curve reconstruction

Since each FXT candidate here has `group_number == 1` (a single observation, no repeat detections), each source needs only **one** light-curve panel -- no multi-observation timeline stitching is required, unlike the QPO/QPE notebooks.

As before, `pn_LC` carries no per-frame timestamps in the published table, so times are reconstructed from the `pn_LC_BB` block boundaries (frames assumed equally spaced within each block).

In [ ]:
def _clean_bb(bb_row):
    arr = np.ma.filled(np.atleast_2d(bb_row), np.nan)
    valid = ~np.isnan(arr).any(axis=1)
    arr = arr[valid]
    return arr if arr.shape[0] > 0 else None


def _reconstruct_frame_times(lc_bb):
    frame_times = []
    for jj in range(lc_bb.shape[0]):
        start, end, n_frames_block = lc_bb[jj, 0], lc_bb[jj, 1], lc_bb[jj, 2]
        n_frames_block = int(n_frames_block) if n_frames_block > 0 else 1
        dt = (end - start) / n_frames_block
        block_times = start + dt * (np.arange(n_frames_block) + 0.5)
        frame_times.append(block_times)
    return np.concatenate(frame_times)


def _prepare_fxt_lc(row):
    """Return (time_raw, lc, bb, rate_bb) for a single FXT row, or None if unusable."""
    bb = _clean_bb(row["pn_LC_BB"])
    if bb is None:
        return None
    bb = bb[np.argsort(bb[:, 0])]

    raw_lc = np.ma.filled(np.atleast_2d(row["pn_LC"]), np.nan)
    mask_lc = ~np.isnan(raw_lc[:, 0])
    lc = raw_lc[mask_lc]
    if lc.size == 0:
        return None

    frame_times = _reconstruct_frame_times(bb)
    n_use = min(len(frame_times), len(lc))
    frame_times = frame_times[:n_use]
    lc = lc[:n_use]

    t0 = bb[0, 0]
    time_raw = frame_times - t0

    rate_bb = bb[:, 3] / np.where(bb[:, 2] == 0, 1, bb[:, 2])
    return time_raw, lc, bb, rate_bb, t0

## 13. Function 1 — plot a single FXT candidate

In [ ]:
def plot_fxt_lightcurve(out, detid, ax=None, show_gold_box=True, save=False):
    """
    Plot the pn raw + Bayesian-block light curve of a single FXT candidate
    (identified by DETID), using only pn_LC/pn_LC_BB from the published
    `out` table. If `ax` is provided, draws into it (for use in a grid);
    otherwise creates a standalone figure.
    """
    rows = out[out["DETID"] == detid]
    if len(rows) == 0:
        print(f"No row found for DETID={detid}")
        return
    row = rows[0]

    prepared = _prepare_fxt_lc(row)
    if prepared is None:
        print(f"No valid light-curve data for DETID={detid}")
        return
    time_raw, lc, bb, rate_bb, t0 = prepared

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(5, 3.5))

    ax.scatter(time_raw, lc[:, 1], color="gray", alpha=0.6, zorder=1, marker="+", s=10)
    ax.plot(time_raw, lc[:, 0], color="red", alpha=0.5, drawstyle="steps-mid", lw=1, zorder=1)
    ax.scatter(time_raw, lc[:, 0], color="red", alpha=0.8, zorder=1, marker="*", s=10)

    for jj in range(bb.shape[0]):
        start = bb[jj, 0] - t0
        end = bb[jj, 1] - t0
        rate = rate_bb[jj]
        is_sig = bb[jj, 5] == 1
        ax.plot([start, end], [rate, rate], color="black" if is_sig else "orange",
                lw=1.5, solid_capstyle="butt", zorder=3)
        if is_sig:
            ax.axvspan(start, end, color="gray", alpha=0.1, zorder=2)

    if show_gold_box and bb.shape[0] == 3 and bb[1, 5] == 1:
        limit = np.median(lc[:, 0]) + 2 * mad(lc[:, 0])
        peaks, _ = signal.find_peaks(lc[:, 0], prominence=limit)
        if len(peaks) >= 1:
            rect = Rectangle((0, 0), 1, 1, transform=ax.transAxes,
                              fill=False, edgecolor="gold", linewidth=2, alpha=0.6, zorder=5)
            ax.add_patch(rect)

    ax.set_title(f"{detid}", fontsize=10)
    ax.tick_params(labelsize=9)

    if standalone:
        ax.set_xlabel(r"Time $-$ $t_0$ [s]", fontsize=10)
        ax.set_ylabel("Counts", fontsize=10)
        plt.tight_layout()
        if save:
            plt.savefig(f"FXT_{detid}.pdf", bbox_inches="tight")
        plt.show()

In [ ]:
# Example: reproduce the paper's 4 example light curves (Fig. sxrts_LCs)
example_detids = [108419004010045, 107821501010059, 108019309010064, 109038936010074]
for detid in example_detids:
    plot_fxt_lightcurve(out, detid)

## 14. Function 2 — grid of all FXT candidates

Lays out every source in a given table as a grid with a fixed number of columns, one panel per source (each source has exactly one observation by construction, since `group_number == 1`).

In [ ]:
def plot_fxt_grid(table, n_cols=4, show_gold_box=True, save_name="fxt_grid_all.pdf"):
    """
    Plot every FXT candidate in `table` as a grid, n_cols wide, with as
    many rows as needed to fit all sources.
    """
    n_total = len(table)
    n_rows = int(np.ceil(n_total / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.2))
    axes = np.atleast_1d(axes).flatten()

    plot_idx = 0
    for row in table:
        if plot_idx >= n_rows * n_cols:
            break
        ax = axes[plot_idx]
        plot_fxt_lightcurve(table, row["DETID"], ax=ax, show_gold_box=show_gold_box)

        if plot_idx % n_cols == 0:
            ax.set_ylabel("Counts", fontsize=9)
        ax.set_xlabel("")
        plot_idx += 1

    for i in range(plot_idx, n_rows * n_cols):
        axes[i].axis("off")

    fig.supxlabel(r"Time $-$ $t_0$ [s]", fontsize=10)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15 / n_rows + 0.05)
    plt.savefig(save_name, bbox_inches="tight")
    plt.show()

In [ ]:
# Full grid: every FXT candidate in the final sample, 4 columns wide
plot_fxt_grid(flashes_extrag_UNI_B10, n_cols=4)